# Task-Conditioned Evaluation of Quantized Small LLMs (BTP GP159)
### Delhi Technological University | B.Tech Major Project (Session 2026–27)
**Team:** Ekansh Agarwal (`23/CS/149`), Faqre Alam (`23/CS/150`), Guneet Toppo (`23/CS/160`)
**Supervisor:** Ms. Gull Kaur, Assistant Professor, Department of CSE, DTU

---
### Kaggle T4 Execution Instructions:
1. **Accelerator:** Under the right panel (Session options), set **Accelerator** to **GPU T4 x1** (16 GB VRAM).
2. **Internet:** Turn **Internet ON**.
3. **Hugging Face Secret:** Go to **Add-ons -> Secrets** and add `HF_TOKEN` containing your Hugging Face read access token (required for gated Llama-3.2-1B-Instruct access).

In [ ]:
# 1. Verify NVIDIA T4 GPU Environment
!nvidia-smi
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")

In [ ]:
# 1b. Set up repository in Kaggle workspace
import os
if not os.path.exists("run_experiments.py"):
    !git clone https://github.com/itsalam149/final-year-btp.git
    %cd final-year-btp
else:
    !git pull origin main


In [ ]:
# 2. Install Pinned Dependencies for Strict Reproducibility
!pip install -q transformers>=4.40.0,<4.45.0 accelerate>=0.28.0
!pip install -q bitsandbytes>=0.43.0 optimum>=1.19.0
!pip install -q lm-eval==0.4.3
!pip install -q langdetect immutabledict
!pip install -q scipy pandas matplotlib tabulate pyyaml tqdm
# Optional: install auto-gptq prebuilt wheel for CUDA
!pip install -q auto-gptq --extra-index-url https://huggingface.github.io/autogptq-index/whl/cu121/ || echo 'AutoGPTQ wheel fallback to CPU/Optimum'

In [ ]:
# 3. Hugging Face Login (for gated Llama-3.2-1B)
import os
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        hf_token = user_secrets.get_secret("HF_TOKEN")
    except Exception:
        pass

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    login(token=hf_token)
    print("✅ Hugging Face Authentication Successful!")
else:
    print("⚠️ No HF_TOKEN found in Kaggle Secrets (Add-ons -> Secrets).")
    print("Qwen and TinyLlama runs will execute; add HF_TOKEN secret to run Llama-3.2.")

In [ ]:
# 4. Verify Grid and Dry-Run
!python run_experiments.py --dry-run

In [ ]:
# 5. Phase B+ Audit: Checkpoint Provenance Verification
!python scripts/audit_checkpoints.py --update-yaml

In [ ]:
# 6. Phase B: Execute FP16 Baselines (Both Models)
!python run_experiments.py --variant fp16

In [ ]:
# 7. Phase C: Execute Core Quantization Runs (8-bit and 4-bit) on Standard Benchmarks
!python run_experiments.py --tasks wikitext,c4,arc_easy,hellaswag,winogrande,mmlu

In [ ]:
# 8. Phase D: Execute Generative Reasoning & Instruction Following (GSM8K & IFEval)
!python run_experiments.py --tasks gsm8k,ifeval

In [ ]:
# 9. Phase E & G: Run Statistical Analysis (10,000 Paired Bootstrap CIs, Spearman Proxy & Bit-Budget)
!python run_analysis.py --csv results/results_master.csv --bootstrap 10000

In [ ]:
# 10. Display Analytical Findings Inline
from IPython.display import Markdown, display
for rpt in ["results/degradation_report.md", "results/spearman_proxy_report.md", "results/bit_budget_guidelines.md"]:
    if os.path.exists(rpt):
        with open(rpt) as f:
            display(Markdown(f.read()))
        print("\n" + "="*75 + "\n")

In [ ]:
# 11. Archive All Results and Sample Logs for Download
!zip -r btp_evaluation_results.zip results/
print("✅ Results successfully zipped into btp_evaluation_results.zip!")